In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# 1. Generate Synthetic Imbalanced B2B Lead Database
np.random.seed(101)
n_leads = 15000

data = pd.DataFrame({
    'company_size_headcount': np.random.randint(10, 5000, n_leads),
    'pricing_page_clicks': np.random.poisson(lam=0.3, size=n_leads),
    'total_downloads': np.random.poisson(lam=0.6, size=n_leads),
    'engagement_velocity': np.random.exponential(scale=0.5, size=n_leads),
    'is_executive_role': np.random.binomial(n=1, p=0.15, size=n_leads)
})

# Create true latent conversion function with 4% conversion base
latent_score = (
    0.0005 * data['company_size_headcount'] +
    1.20 * data['pricing_page_clicks'] +
    0.80 * data['total_downloads'] +
    2.10 * data['engagement_velocity'] +
    1.50 * data['is_executive_role'] - 4.5
)
prob_conversion = 1 / (1 + np.exp(-latent_score))
data['converted_sql'] = np.random.binomial(n=1, p=prob_conversion)

# 2. Pipeline Preprocessing & Train Split
X = data.drop(columns=['converted_sql'])
y = data['converted_sql']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y)

# 3. Model Training with Scale_Pos_Weight for Class Imbalance
imbalance_ratio = (len(y_train) - y_train.sum()) / y_train.sum()
model = XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    learning_rate=0.08,
    max_depth=5,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

# 4. Calibration Output Validation
preds_prob = model.predict_proba(X_test)[:, 1]
# Operational optimization threshold set to 0.40 instead of 0.50
custom_preds = (preds_prob >= 0.40).astype(int)

print(f"ROC AUC Metric Performance: {roc_auc_score(y_test, preds_prob):.4f}")
print("\n--- Classification Matrix Output ---")
print(classification_report(y_test, custom_preds))


ROC AUC Metric Performance: 0.8334

--- Classification Matrix Output ---
              precision    recall  f1-score   support

           0       0.90      0.67      0.77      2091
           1       0.52      0.82      0.64       909

    accuracy                           0.71      3000
   macro avg       0.71      0.74      0.70      3000
weighted avg       0.78      0.71      0.73      3000

